# TF-IDF 从零实现示例

**TF-IDF**（Term Frequency - Inverse Document Frequency）用一个实数衡量一个词对一篇文档的「重要性」：在某篇文档里出现越多越重要（TF），但在整个语料里越常见越不重要（IDF）。

本笔记用**纯 Python 标准库**从零实现 TF-IDF，并与 `scikit-learn` 的 `TfidfVectorizer` 做对照。

公式：
- $TF(t,d) = \dfrac{\text{词 } t \text{ 在文档 } d \text{ 中的出现次数}}{\text{文档 } d \text{ 的总词数}}$
- $IDF(t) = \ln\!\left(\dfrac{1 + N}{1 + n_t}\right) + 1$  （`smooth_idf` 形式）
- $TF\text{-}IDF(t,d) = TF(t,d) \times IDF(t)$

In [1]:
import re
import math
from collections import Counter

def tokenize(text):
    # 简单英文分词 + 小写；中文场景可换成 jieba.cut
    return re.findall(r"[a-z0-9]+", text.lower())

print("tokenize 示例:", tokenize("The cat sat on the MAT!"))

tokenize 示例: ['the', 'cat', 'sat', 'on', 'the', 'mat']


## 1. 准备语料

四篇小文档，用来演示计算过程。

In [2]:
corpus = [
    "The cat sat on the mat",
    "The dog played in the mat",
    "Cats and dogs are pets",
    "The bird sang a song",
]
docs = [tokenize(d) for d in corpus]
for i, d in enumerate(docs):
    print(f"doc{i}: {d}")

doc0: ['the', 'cat', 'sat', 'on', 'the', 'mat']
doc1: ['the', 'dog', 'played', 'in', 'the', 'mat']
doc2: ['cats', 'and', 'dogs', 'are', 'pets']
doc3: ['the', 'bird', 'sang', 'a', 'song']


## 2. 词频 TF

对每篇文档，统计每个词的出现频率（归一化到该文档词数的比例）。

In [3]:
def term_frequency(tokens):
    counts = Counter(tokens)
    total = len(tokens)
    return {t: c / total for t, c in counts.items()}

tf = [term_frequency(d) for d in docs]
print("doc0 的 TF:", tf[0])

doc0 的 TF: {'the': 0.3333333333333333, 'cat': 0.16666666666666666, 'sat': 0.16666666666666666, 'on': 0.16666666666666666, 'mat': 0.16666666666666666}


## 3. 逆文档频率 IDF

统计每个词出现在多少篇文档中（文档频率 $n_t$），再换算成 IDF。

In [4]:
def compute_idf(docs):
    N = len(docs)
    df = {}
    for d in docs:
        for t in set(d):          # 同一文档只计一次
            df[t] = df.get(t, 0) + 1
    # smooth_idf 形式，与 sklearn 默认一致
    return {t: math.log((1 + N) / (1 + n)) + 1 for t, n in df.items()}

idf_scores = compute_idf(docs)
for t in sorted(idf_scores):
    print(f"{t:8s} idf={idf_scores[t]:.4f}")

a        idf=1.9163
and      idf=1.9163
are      idf=1.9163
bird     idf=1.9163
cat      idf=1.9163
cats     idf=1.9163
dog      idf=1.9163
dogs     idf=1.9163
in       idf=1.9163
mat      idf=1.5108
on       idf=1.9163
pets     idf=1.9163
played   idf=1.9163
sang     idf=1.9163
sat      idf=1.9163
song     idf=1.9163
the      idf=1.2231


In [6]:
print(idf_scores)

{'mat': 1.5108256237659907, 'sat': 1.916290731874155, 'the': 1.2231435513142097, 'on': 1.916290731874155, 'cat': 1.916290731874155, 'played': 1.916290731874155, 'in': 1.916290731874155, 'dog': 1.916290731874155, 'and': 1.916290731874155, 'are': 1.916290731874155, 'dogs': 1.916290731874155, 'pets': 1.916290731874155, 'cats': 1.916290731874155, 'song': 1.916290731874155, 'bird': 1.916290731874155, 'sang': 1.916290731874155, 'a': 1.916290731874155}


## 4. 计算 TF-IDF 并展示矩阵

把每篇文档表示成「词表维度」的向量，缺失词补 0。

矩阵横轴上的是一个个的词 term，

纵轴上的是一个个的文档 doc，

矩阵的值表示的含义是某个词在这篇文档里的重要程度。词在当前文档出现越多，在全局文档越罕见，那么这个值就会越大。

In [5]:
def tfidf(tf_doc, idf_scores):
    return {t: tf_doc.get(t, 0.0) * idf_scores[t] for t in idf_scores}

tfidf_matrix = [tfidf(tf_d, idf_scores) for tf_d in tf]

vocab = sorted(idf_scores.keys())
header = "term".ljust(10) + "".join(f"{t:>10}" for t in vocab)
print(header)
for i, row in enumerate(tfidf_matrix):
    line = f"doc{i}".ljust(10) + "".join(f"{row.get(t, 0):>10.3f}" for t in vocab)
    print(line)

term               a       and       are      bird       cat      cats       dog      dogs        in       mat        on      pets    played      sang       sat      song       the
doc0           0.000     0.000     0.000     0.000     0.319     0.000     0.000     0.000     0.000     0.252     0.319     0.000     0.000     0.000     0.319     0.000     0.408
doc1           0.000     0.000     0.000     0.000     0.000     0.000     0.319     0.000     0.319     0.252     0.000     0.000     0.319     0.000     0.000     0.000     0.408
doc2           0.000     0.383     0.383     0.000     0.000     0.383     0.000     0.383     0.000     0.000     0.000     0.383     0.000     0.000     0.000     0.000     0.000
doc3           0.383     0.000     0.000     0.383     0.000     0.000     0.000     0.000     0.000     0.000     0.000     0.000     0.000     0.383     0.000     0.383     0.245


## 5. 与 scikit-learn 对照（可选）

如果环境装了 `scikit-learn`，下面会输出官方实现的结果，方便验证我们手写的版本。

In [6]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer()
    X = vec.fit_transform(corpus)
    print("词表:", vec.get_feature_names_out())
    print("矩阵形状:", X.shape)
    print("(sklearn 默认会对每篇文档做 L2 归一化，所以数值与我们未归一化的版本略有差异)")
except ImportError:
    print("未安装 scikit-learn，跳过对照。安装：pip install scikit-learn")

词表: ['and' 'are' 'bird' 'cat' 'cats' 'dog' 'dogs' 'in' 'mat' 'on' 'pets'
 'played' 'sang' 'sat' 'song' 'the']
矩阵形状: (4, 16)
(sklearn 默认会对每篇文档做 L2 归一化，所以数值与我们未归一化的版本略有差异)


## 6. 用 TF-IDF 做检索（搜索）

TF-IDF 把文档和查询都映射到同一套「词表维度」的向量空间，因此检索就变成**向量相似度比较**：
把查询也转成 TF-IDF 向量，计算它与每篇文档向量的**余弦相似度**，按相似度降序返回 Top-K。

这一步正是「稀疏检索」的核心环节——前面几节完成了「建向量」，这里补上「拿查询去匹配文档并排序」。

In [8]:
import math

def cosine_similarity(vec_a, vec_b):
    # 两个稀疏向量（dict 形式）的余弦相似度
    dot = sum(vec_a[t] * vec_b[t] for t in set(vec_a) & set(vec_b))
    norm_a = math.sqrt(sum(v * v for v in vec_a.values()))
    norm_b = math.sqrt(sum(v * v for v in vec_b.values()))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def search(query, tfidf_matrix, corpus, idf_scores, top_k=3):
    q_tokens = tokenize(query)
    q_vec = tfidf(term_frequency(q_tokens), idf_scores)  # 查询也转成 TF-IDF 向量
    scored = []
    print(f"q_vec: {q_vec}");
    print("d_vec", tfidf_matrix[0]);
    for i, d_vec in enumerate(tfidf_matrix):
        scored.append((i, cosine_similarity(q_vec, d_vec)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

for q in ["cat mat", "dog pets", "bird song"]:
    print(f'\n查询: "{q}"')
    for i, s in search(q, tfidf_matrix, corpus, idf_scores):
        print(f"  doc{i}  sim={s:.4f}   {corpus[i]}")


查询: "cat mat"
q_vec: {'mat': 0.7554128118829954, 'sat': 0.0, 'the': 0.0, 'on': 0.0, 'cat': 0.9581453659370776, 'played': 0.0, 'in': 0.0, 'dog': 0.0, 'and': 0.0, 'are': 0.0, 'dogs': 0.0, 'pets': 0.0, 'cats': 0.0, 'song': 0.0, 'bird': 0.0, 'sang': 0.0, 'a': 0.0}
d_vec {'mat': 0.2518042706276651, 'sat': 0.3193817886456925, 'the': 0.40771451710473655, 'on': 0.3193817886456925, 'cat': 0.3193817886456925, 'played': 0.0, 'in': 0.0, 'dog': 0.0, 'and': 0.0, 'are': 0.0, 'dogs': 0.0, 'pets': 0.0, 'cats': 0.0, 'song': 0.0, 'bird': 0.0, 'sang': 0.0, 'a': 0.0}
  doc0  sim=0.5557   The cat sat on the mat
  doc1  sim=0.2130   The dog played in the mat
  doc2  sim=0.0000   Cats and dogs are pets

查询: "dog pets"
q_vec: {'mat': 0.0, 'sat': 0.0, 'the': 0.0, 'on': 0.0, 'cat': 0.0, 'played': 0.0, 'in': 0.0, 'dog': 0.9581453659370776, 'and': 0.0, 'are': 0.0, 'dogs': 0.0, 'pets': 0.9581453659370776, 'cats': 0.0, 'song': 0.0, 'bird': 0.0, 'sang': 0.0, 'a': 0.0}
d_vec {'mat': 0.2518042706276651, 'sat': 0.31938

## 小结

- `the`、`a` 这类常见词 IDF 低，对区分文档贡献小；
- `cat`、`dog` 这类有区分度的词权重更高；
- 得到的 TF-IDF 向量就是**稀疏向量**（大部分维度为 0），可直接用于「稀疏检索」中的相似度计算（如余弦相似度）。